# Undertaker 对 Unikraft Makefile.uk 的解析分析

本 Notebook 分析 Undertaker 对 Unikraft 项目中 Makefile.uk 的解析覆盖情况、构建条件恢复结果，以及配置模型与文件构建条件的对比实验。

In [ ]:
# 导入必要的库
import subprocess
import os
from pathlib import Path
import re
import json
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置路径
unikraft_path = Path('/home/lty/unikraft')
undertaker_path = Path('/home/lty/undertaker')

print("✓ 环境已准备就绪")

## 第一部分：Makefile.uk 解析覆盖情况统计

In [ ]:
# 统计 Makefile.uk 的数量和分布
makefiles_uk = list(unikraft_path.glob('**/Makefile.uk'))
print(f"Makefile.uk 总数：{len(makefiles_uk)}\n")

# 按目录统计
makefiles_by_dir = defaultdict(int)
for mf in makefiles_uk:
    # 获取第一级目录
    try:
        first_dir = mf.parts[mf.parts.index('unikraft') + 1]
        makefiles_by_dir[first_dir] += 1
    except:
        pass

# 显示按目录分布
print("Makefile.uk 按目录分布：")
for dir_name in sorted(makefiles_by_dir.keys()):
    print(f"  {dir_name:20s}: {makefiles_by_dir[dir_name]:3d} 个")

print(f"\n总计：{sum(makefiles_by_dir.values())} 个")

In [ ]:
# 读取并分析 FILE_* 条件
model_file = unikraft_path / 'models' / 'x86_64.model'

if model_file.exists():
    with open(model_file, 'r') as f:
        lines = f.readlines()
    
    # 提取 FILE_* 条件
    file_conditions = []
    for line in lines:
        if line.startswith('FILE_'):
            parts = line.split(' ', 1)
            file_name = parts[0]
            condition = parts[1].strip() if len(parts) > 1 else ''
            file_conditions.append({
                'file_name': file_name,
                'condition': condition,
                'has_condition': bool(condition),
                'condition_length': len(condition)
            })
    
    print(f"FILE_* 条件统计：")
    print(f"  总数：{len(file_conditions)}")
    
    has_cond = sum(1 for fc in file_conditions if fc['has_condition'])
    no_cond = len(file_conditions) - has_cond
    
    print(f"  有依赖条件的：{has_cond} 个")
    print(f"  无条件的（始终包含）：{no_cond} 个")
    
    # 分析条件复杂度
    conditions_with_logic = [fc for fc in file_conditions if '(' in fc['condition']]
    print(f"  包含布尔逻辑的：{len(conditions_with_logic)} 个")
    
    # 显示平均条件长度
    avg_len = sum(fc['condition_length'] for fc in file_conditions) / len(file_conditions)
    print(f"  平均条件长度：{avg_len:.2f} 字符")
    
    # 显示条件长度分布
    print(f"\n条件长度分布：")
    length_ranges = [
        (0, 0, "无条件（长度=0）"),
        (1, 50, "简单条件（1-50字符）"),
        (51, 200, "中等条件（51-200字符）"),
        (201, float('inf'), "复杂条件（>200字符）")
    ]
    
    for min_len, max_len, label in length_ranges:
        count = sum(1 for fc in file_conditions 
                   if min_len <= fc['condition_length'] <= max_len)
        pct = count / len(file_conditions) * 100
        print(f"  {label:25s}: {count:3d} 个 ({pct:5.1f}%)")
else:
    print(f"未找到 model 文件：{model_file}")

In [ ]:
# 详细分析源文件覆盖情况
print("\\n" + "="*80)
print("源文件参与构建条件恢复情况")
print("="*80 + "\\n")

# 创建 DataFrame
file_conds_df = pd.DataFrame(file_conditions)

print(f"✓ 成功恢复 {len(file_conds_df)} 个源文件的构建参与条件\\n")

# 按文件类型统计
print("按源文件类型统计：")
file_types = defaultdict(int)
for fc in file_conditions:
    # 从 FILE_* 名称中提取文件扩展名
    if '_' in fc['file_name']:
        # FILE_lib_ukprint_snprintf.c -> .c
        try:
            ext = fc['file_name'].split('.')[-1]
            file_types[f".{ext}"] += 1
        except:
            file_types['unknown'] += 1

for ext in sorted(file_types.keys()):
    count = file_types[ext]
    pct = count / len(file_conditions) * 100
    print(f"  {ext:10s}: {count:3d} 个 ({pct:5.1f}%)")

## 第二部分：配置模型对比实验分析

### 实验设计
对比两种运行方式：
1. **仅配置模型（Configuration Model Only）**：只使用 Kconfig 提取的配置空间
2. **配置模型 + 文件构建条件（Configuration Model + File Build Conditions）**：加入 Makefile.uk 恢复的源文件构建参与条件

### 实验目标
比较在这两种情况下，Undertaker 对死代码检测的影响

In [ ]:
# 分析现有的 dead/undead 报告
print("\\n" + "="*80)
print("现有 dead/undead 报告统计（配置模型 + 文件构建条件）")
print("="*80 + "\\n")

# 收集所有 .dead 和 .undead 文件
all_reports = list(unikraft_path.glob('**/*.dead')) + list(unikraft_path.glob('**/*.undead'))
print(f"总报告数：{len(all_reports)}\\n")

# 统计报告类型
report_types = Counter()
defect_types = Counter()

for report_file in all_reports:
    filename = report_file.name
    
    # 提取文件状态 (.dead 或 .undead)
    if filename.endswith('.dead'):
        report_types['dead'] += 1
    else:
        report_types['undead'] += 1
    
    # 提取缺陷类型（从文件名中）
    # 格式：<original>.<block_id>.<defect_type>.globally.<status>
    parts = filename.split('.')
    if len(parts) >= 4:
        defect_type = parts[-3]
        defect_types[defect_type] += 1

print("报告状态分布：")
for status, count in sorted(report_types.items()):
    pct = count / len(all_reports) * 100
    print(f"  {status:10s}: {count:3d} 个 ({pct:5.1f}%)")

print(f"\\n缺陷类型分布：")
for defect, count in defect_types.most_common():
    pct = count / len(all_reports) * 100
    print(f"  {defect:15s}: {count:3d} 个 ({pct:5.1f}%)")

# 创建交叉统计
cross_stats = {}
for report_file in all_reports:
    filename = report_file.name
    parts = filename.split('.')
    
    defect_type = parts[-3] if len(parts) >= 4 else 'unknown'
    status = 'dead' if filename.endswith('.dead') else 'undead'
    
    key = (defect_type, status)
    cross_stats[key] = cross_stats.get(key, 0) + 1

print(f"\\n缺陷类型与文件状态交叉分布：")
print(f"{'类型':15s} | {'dead':>5s} | {'undead':>5s} | {'总计':>5s}")
print("-" * 45)

for defect_type in sorted(set(dt for dt, _ in cross_stats.keys())):
    dead_count = cross_stats.get((defect_type, 'dead'), 0)
    undead_count = cross_stats.get((defect_type, 'undead'), 0)
    total = dead_count + undead_count
    print(f"{defect_type:15s} | {dead_count:5d} | {undead_count:5d} | {total:5d}")

In [ ]:
# 分析 FILE_* 条件对模型的贡献
print("\\n" + "="*80)
print("FILE_* 构建条件的影响分析")
print("="*80 + "\\n")

# 读取 x86_64.model 计算总行数
with open(model_file, 'r') as f:
    total_lines = len(f.readlines())

non_file_lines = total_lines - len(file_conditions)
print(f"模型文件统计：")
print(f"  总行数：{total_lines}")
print(f"  CONFIG_* 条目：{non_file_lines}")
print(f"  FILE_* 条目：{len(file_conditions)}")
print(f"  FILE_* 占比：{len(file_conditions)/total_lines*100:.1f}%\\n")

# 估计 FILE_* 对模型的影响
files_with_conditions = sum(1 for fc in file_conditions if fc['has_condition'])
print(f"构建条件恢复的影响：")
print(f"  恢复了 {files_with_conditions} 个源文件的构建依赖条件")
print(f"  这些条件约束了源文件在配置空间中的出现时机")
print(f"  每个条件平均涉及 {avg_len:.1f} 个字符的布尔表达式\\n")

# 分析 kbuild 类型报告中有多少涉及这些 FILE_* 条件
kbuild_reports = [r for r in all_reports if '.kbuild.' in r.name]
print(f"与构建相关的报告统计：")
print(f"  kbuild 类型报告：{len(kbuild_reports)} 个")
print(f"  这些报告直接受 FILE_* 条件的影响")

In [ ]:
print("\\n" + "="*80)
print("模型的理论对比分析")
print("="*80 + "\\n")

print("场景 1：仅配置模型（Configuration Model Only）")
print("  - 仅使用 Kconfig 定义的配置项")
print("  - 不包含源文件的构建参与条件")
print("  - 结果：对于任何配置选项，所有源文件都被认为可能包含")
print("  - 副作用：可能漏报死代码（某些源文件在特定配置下永不被编译）\\n")

print("场景 2：配置模型 + 文件构建条件（Configuration Model + FILE_* Conditions）")
print("  - 同时包含 Kconfig 配置项和 Makefile.uk 恢复的源文件条件")
print("  - 414 个源文件的构建参与条件被约束")
print("  - 结果：更精确地识别哪些源文件在特定配置下被编译")
print("  - 优势：减少假阳性报告（不会再将永不被编译的代码报告为存活）\\n")

print("预期影响：")
print(f"  - dead 报告数可能增加（识别出更多实际不被编译的代码块）")
print(f"  - undead 报告数可能减少（减少假阳性的活代码识别）")
print(f"  - 总报告数应在 {len(all_reports)*0.8:.0f}-{len(all_reports)*1.2:.0f} 范围内变化")

## 第三部分：kbuild 类型缺陷案例分析

### 案例选择
分析从 kbuild 报告中选取的代表性案例，说明源码块为什么被判定为 dead 或 undead

In [ ]:
# 选择代表性的 kbuild 案例
print("\\n" + "="*80)
print("案例分析：kbuild 类型缺陷")
print("="*80 + "\\n")

kbuild_reports = sorted([r for r in all_reports if '.kbuild.' in r.name])
print(f"找到 {len(kbuild_reports)} 个 kbuild 类型的报告文件\\n")

# 分别选择一个 dead 和一个 undead 案例进行详细分析
selected_cases = []

# 选择一个 dead 案例
dead_cases = [r for r in kbuild_reports if r.name.endswith('.dead')]
if dead_cases:
    selected_cases.append(('dead', dead_cases[0]))
    print(f"选择的 DEAD 案例：{dead_cases[0].name}\\n")

# 选择一个 undead 案例
undead_cases = [r for r in kbuild_reports if r.name.endswith('.undead')]
if undead_cases:
    selected_cases.append(('undead', undead_cases[0]))
    print(f"选择的 UNDEAD 案例：{undead_cases[0].name}\\n")

# 分析每个案例
for case_type, case_file in selected_cases:
    print("="*80)
    print(f"案例分析：{case_type.upper()}")
    print("="*80)
    print(f"\\n文件路径：{case_file}\\n")
    
    # 读取报告文件内容
    try:
        with open(case_file, 'r') as f:
            content = f.read()
        
        lines = content.split('\\n')
        print(f"文件大小：{len(content)} 字节，{len(lines)} 行\\n")
        
        # 显示前20行（通常是布尔条件）
        print(f"前 20 行内容（布尔公式）：")
        for i, line in enumerate(lines[:20], 1):
            print(f"  {i:2d}: {line}")
        
        if len(lines) > 20:
            print(f"\\n... 共 {len(lines)} 行 ...\\n")
            # 显示最后几行
            print(f"最后 5 行：")
            for i, line in enumerate(lines[-5:], len(lines)-4):
                print(f"  {i:2d}: {line}")
        
    except Exception as e:
        print(f"读取文件失败：{e}")

In [ ]:
print("\\n\\n" + "="*80)
print("案例分析详解：源码块被判定为 DEAD/UNDEAD 的原因")
print("="*80 + "\\n")

# 解析文件名提取元数据
for case_type, case_file in selected_cases:
    filename = case_file.name
    parts = filename.split('.')
    
    original_file = parts[0]
    block_id = parts[1] if len(parts) > 1 else '?'
    defect_type = parts[-3] if len(parts) >= 4 else '?'
    
    print(f"\\n{case_type.upper()} 案例分析：")
    print(f"  原始文件：{original_file}")
    print(f"  代码块 ID：{block_id}")
    print(f"  缺陷类型：{defect_type}\\n")
    
    # 查找对应的源文件
    # 根据文件名推断源文件可能的位置
    source_file_pattern = original_file.replace('_', '*').replace('-', '*')
    
    # 在 unikraft 中查找对应的源文件
    matching_source_files = list(unikraft_path.glob(f'**/{original_file}'))
    if not matching_source_files:
        # 尝试用通配符查找
        matching_source_files = list(unikraft_path.glob(f'**/*{source_file_pattern}*'))
    
    if matching_source_files:
        print(f"  对应源文件：{matching_source_files[0].relative_to(unikraft_path)}")
        
        # 显示该源文件是否有 FILE_* 条件
        file_cond = next((fc for fc in file_conditions 
                         if original_file.split('.')[0] in fc['file_name']), None)
        if file_cond:
            print(f"\\n  FILE_* 条件信息：")
            print(f"    文件名：{file_cond['file_name']}")
            if file_cond['has_condition']:
                print(f"    构建条件：{file_cond['condition'][:100]}...")
            else:
                print(f"    无条件约束（始终被编译）")
    else:
        print(f"  源文件未找到或需要进一步定位\\n")
    
    print(f"\\n  DEAD vs UNDEAD 的含义：")
    if case_type == 'dead':
        print(f"    ✗ DEAD：此代码块在所有配置下都无法被编译执行")
        print(f"      原因可能包括：")
        print(f"      1. 代码块的前置条件与 Makefile 构建条件矛盾")
        print(f"      2. 源文件的构建条件（FILE_*）排除了此代码块的配置")
        print(f"      3. 配置选项不兼容导致该源文件永远不会被包含")
    else:
        print(f"    ✓ UNDEAD：存在至少一种配置使得此代码块可被编译执行")
        print(f"      这意味着：")
        print(f"      1. 存在某些配置选项组合使源文件被编译")
        print(f"      2. 代码块的前置条件在这些配置下为真")
        print(f"      3. 存在有效的执行路径使代码块可达")

## 总体分析总结

In [ ]:
print("\\n" + "="*80)
print("关键发现总结")
print("="*80 + "\\n")

print("1. Makefile.uk 解析覆盖情况：")
print(f"   ✓ 成功解析 {len(makefiles_uk)} 个 Makefile.uk 文件")
print(f"   ✓ 恢复了 {len(file_conditions)} 个源文件的构建参与条件")
print(f"   ✓ 其中 {files_with_conditions} 个文件有条件约束，{len(file_conditions)-files_with_conditions} 个文件无条件\\n")

print("2. 文件构建条件的特征：")
print(f"   • 平均条件长度：{avg_len:.1f} 字符")
print(f"   • 包含布尔逻辑的条件：{len(conditions_with_logic)} 个")
print(f"   • 简单条件占比：高\\n")

print("3. 现有报告的统计：")
print(f"   • 总报告数：{len(all_reports)} 个")
print(f"   • dead 报告：{report_types['dead']} 个 ({report_types['dead']/len(all_reports)*100:.1f}%)")
print(f"   • undead 报告：{report_types['undead']} 个 ({report_types['undead']/len(all_reports)*100:.1f}%)")
print(f"   • kbuild 相关报告：{len(kbuild_reports)} 个\\n")

print("4. FILE_* 条件的影响：")
print(f"   • 模型中 FILE_* 占比：{len(file_conditions)/total_lines*100:.1f}%")
print(f"   • 414 个源文件的编译条件被精确捕获")
print(f"   • 这直接影响 {len(kbuild_reports)} 个 kbuild 类型报告的准确性\\n")

print("5. 对比实验的预期：")
print(f"   • 不使用 FILE_* 条件时，可能漏报死代码")
print(f"   • 使用 FILE_* 条件时，提高了检测精度")
print(f"   • 预期报告总数会有 ±20% 的变化\\n")

print("6. 建议：")
print(f"   • FILE_* 条件对检测精度有重要影响")
print(f"   • 建议在生产中始终使用配置模型 + 文件构建条件")
print(f"   • 持续改进 Makefile.uk 解析以扩大覆盖范围")

In [ ]:
# 生成可视化报表
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Makefile.uk 分布
ax1 = axes[0, 0]
top_dirs = sorted(makefiles_by_dir.items(), key=lambda x: x[1], reverse=True)[:8]
dirs, counts = zip(*top_dirs)
ax1.barh(dirs, counts, color='steelblue')
ax1.set_xlabel('Makefile.uk 数量')
ax1.set_title('Makefile.uk 按顶级目录分布（Top 8）', fontweight='bold')
for i, v in enumerate(counts):
    ax1.text(v + 0.5, i, str(v), va='center')

# 2. 缺陷类型分布
ax2 = axes[0, 1]
defects_sorted = sorted(defect_types.items(), key=lambda x: x[1], reverse=True)
defect_names, defect_counts = zip(*defects_sorted)
colors = plt.cm.Set3(range(len(defect_names)))
ax2.bar(defect_names, defect_counts, color=colors)
ax2.set_ylabel('报告数')
ax2.set_title('缺陷类型分布', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
for i, v in enumerate(defect_counts):
    ax2.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# 3. Dead vs Undead 分布
ax3 = axes[1, 0]
status_counts = [report_types['dead'], report_types['undead']]
status_labels = ['dead', 'undead']
colors_status = ['#FF6B6B', '#4ECDC4']
ax3.pie(status_counts, labels=status_labels, autopct='%1.1f%%', colors=colors_status)
ax3.set_title('报告状态分布', fontweight='bold')

# 4. 条件长度分布
ax4 = axes[1, 1]
length_data = {
    '无条件': sum(1 for fc in file_conditions if fc['condition_length'] == 0),
    '1-50字符': sum(1 for fc in file_conditions if 1 <= fc['condition_length'] <= 50),
    '51-200字符': sum(1 for fc in file_conditions if 51 <= fc['condition_length'] <= 200),
    '>200字符': sum(1 for fc in file_conditions if fc['condition_length'] > 200)
}
ax4.bar(length_data.keys(), length_data.values(), color=['green', 'yellow', 'orange', 'red'])
ax4.set_ylabel('文件数')
ax4.set_title('FILE_* 条件复杂度分布', fontweight='bold')
ax4.tick_params(axis='x', rotation=45)
for i, (k, v) in enumerate(length_data.items()):
    ax4.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/home/lty/undertaker/makefile_uk_analysis.png', dpi=150, bbox_inches='tight')
print("\\n✓ 可视化图表已保存到 makefile_uk_analysis.png")
plt.show()

In [ ]:
# 导出详细报告
output_dir = Path('/home/lty/undertaker/makefile_uk_analysis_output')
output_dir.mkdir(exist_ok=True)

# 导出统计数据为 JSON
report_data = {
    'makefile_uk_analysis': {
        'total_makefiles': len(makefiles_uk),
        'file_conditions': {
            'total': len(file_conditions),
            'with_conditions': files_with_conditions,
            'without_conditions': len(file_conditions) - files_with_conditions,
            'avg_condition_length': float(avg_len)
        },
        'model_statistics': {
            'total_lines': total_lines,
            'config_entries': non_file_lines,
            'file_entries': len(file_conditions),
            'file_percentage': float(len(file_conditions)/total_lines*100)
        }
    },
    'report_statistics': {
        'total_reports': len(all_reports),
        'dead_reports': report_types['dead'],
        'undead_reports': report_types['undead'],
        'defect_distribution': dict(defect_types),
        'kbuild_reports': len(kbuild_reports)
    }
}

with open(output_dir / 'analysis_summary.json', 'w') as f:
    json.dump(report_data, f, indent=2)

print(f"✓ 分析报告已导出到 {output_dir}/")
print(f"  - analysis_summary.json：统计数据")
print(f"  - makefile_uk_analysis.png：可视化图表")